# Steganalysis Model Diagnosis Notebook

## Problem Statement
Steganographic images are being misclassified as non-steganographic (showing as "not having" steganography when they do).

## Root Cause Investigation
This notebook systematically analyzes:
1. Model architecture and predictions
2. Confidence score distributions
3. Threshold calibration issues
4. Why images fall into the "uncertain" zone

## Section 1: Import Required Libraries and Load Model

In [ ]:
import sys
from pathlib import Path
import os

# Add backend to path
BACKEND_ROOT = Path.cwd().parent
if BACKEND_ROOT not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
import json
from typing import Any

print(f"Backend root: {BACKEND_ROOT}")
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Import the stego scanner
try:
    from app.scanners.stg.scanner import (
        load_models,
        scan_file,
        scanner_status,
        DEFAULT_CLEAN_THRESHOLD,
        DEFAULT_STEGO_THRESHOLD,
        MODEL_PATH,
    )
    print("✓ Successfully imported stego scanner")
except Exception as e:
    print(f"✗ Error importing stego scanner: {e}")
    import traceback
    traceback.print_exc()

# Check model status
status = scanner_status()
print("\n" + "="*60)
print("MODEL STATUS")
print("="*60)
print(json.dumps(status, indent=2))


## Section 2: Analyze Current Threshold Settings

In [ ]:
print("="*70)
print("THRESHOLD ANALYSIS - THE ROOT CAUSE OF MISCLASSIFICATION")
print("="*70)

print(f"\nCurrent Configuration:")
print(f"  CLEAN_THRESHOLD:  {DEFAULT_CLEAN_THRESHOLD}")
print(f"  STEGO_THRESHOLD:  {DEFAULT_STEGO_THRESHOLD}")

print(f"\nDecision Logic:")
print(f"  IF score <= {DEFAULT_CLEAN_THRESHOLD}:        → COVER (not stego)")
print(f"  IF score >= {DEFAULT_STEGO_THRESHOLD}:        → STEGO (has stego)")
print(f"  ELSE ({DEFAULT_CLEAN_THRESHOLD} < score < {DEFAULT_STEGO_THRESHOLD}): → UNCERTAIN")

gap = DEFAULT_STEGO_THRESHOLD - DEFAULT_CLEAN_THRESHOLD
print(f"\n⚠️  CRITICAL ISSUE: There's a {gap:.2f} point UNCERTAINTY GAP!")
print(f"    Any stego images scoring between {DEFAULT_CLEAN_THRESHOLD} and {DEFAULT_STEGO_THRESHOLD}")
print(f"    will be classified as UNCERTAIN ❌ (false negative!)")

print(f"\n✗ YOUR PROBLEM: Stego images are getting scores in this range!")


## Section 3: Test on Known Stego and Cover Images

In [ ]:
# Find test data
test_dir = Path(r"c:\Users\JOSEPH\Desktop\Sem VIII\Project\steganalysis-gbrasnet\data\GBRASNET")
cover_dir = test_dir / "cover"
stego_dir = test_dir / "stego"

supported_exts = {".jpg", ".jpeg", ".jfif", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

print(f"Test data directory: {test_dir}")
print(f"  Cover images available: {cover_dir.exists()}")
print(f"  Stego images available: {stego_dir.exists()}")

# Collect scores
cover_scores = []
stego_scores = []

print("\n" + "="*70)
print("SCANNING COVER IMAGES (Expected: Low Scores)")
print("="*70)

for img_path in sorted(list(cover_dir.iterdir())[:20]):  # First 20
    if img_path.is_file() and img_path.suffix.lower() in supported_exts:
        try:
            result = scan_file(img_path, log_event=False)
            score = result.get("score", 0.5)
            decision = result.get("decision", "?")
            cover_scores.append(score)
            status_icon = "✓" if decision == "COVER" else "⚠" if decision == "UNCERTAIN" else "✗"
            print(f"{status_icon} {img_path.name:40s} Score: {score:.4f} → {decision}")
        except Exception as e:
            print(f"✗ {img_path.name}: {e}")

print("\n" + "="*70)
print("SCANNING STEGO IMAGES (Expected: High Scores)")
print("="*70)

for img_path in sorted(list(stego_dir.iterdir())[:20]):  # First 20
    if img_path.is_file() and img_path.suffix.lower() in supported_exts:
        try:
            result = scan_file(img_path, log_event=False)
            score = result.get("score", 0.5)
            decision = result.get("decision", "?")
            stego_scores.append(score)
            status_icon = "✓" if decision == "STEGO" else "⚠" if decision == "UNCERTAIN" else "✗"
            print(f"{status_icon} {img_path.name:40s} Score: {score:.4f} → {decision}")
        except Exception as e:
            print(f"✗ {img_path.name}: {e}")

print(f"\nScored {len(cover_scores)} cover images and {len(stego_scores)} stego images")


## Section 4: Analyze Score Distributions

In [ ]:
cd "C:\Users\JOSEPH\Desktop\Sem VIII\Project\CyberShield Innovators\cloud\Backend"
python scripts/calibrate_stg_detector.py ^
  --clean-dir "C:\Users\JOSEPH\Desktop\Sem VIII\Project\steganalysis-gbrasnet\data\GBRASNET\cover" ^
  --stego-dir "C:\Users\JOSEPH\Desktop\Sem VIII\Project\steganalysis-gbrasnet\data\GBRASNET\stego"

In [ ]:
cover_arr = np.asarray(cover_scores)
stego_arr = np.asarray(stego_scores)

print("="*70)
print("SCORE DISTRIBUTION ANALYSIS")
print("="*70)

print(f"\nCOVER Images (Expected: LOW scores, < {DEFAULT_CLEAN_THRESHOLD})")
print(f"  Count:   {len(cover_scores)}")
print(f"  Min:     {cover_arr.min():.4f}")
print(f"  Max:     {cover_arr.max():.4f}")
print(f"  Mean:    {cover_arr.mean():.4f} ← IMPORTANT!")
print(f"  Median:  {np.median(cover_arr):.4f}")
print(f"  Std:     {cover_arr.std():.4f}")

correct_cover = np.sum(cover_arr <= DEFAULT_CLEAN_THRESHOLD)
print(f"\n  Correctly classified as COVER: {correct_cover}/{len(cover_scores)} ({100*correct_cover/len(cover_scores):.1f}%)")

print(f"\n\nSTEGO Images (Expected: HIGH scores, >= {DEFAULT_STEGO_THRESHOLD})")
print(f"  Count:   {len(stego_scores)}")
print(f"  Min:     {stego_arr.min():.4f}")
print(f"  Max:     {stego_arr.max():.4f}")
print(f"  Mean:    {stego_arr.mean():.4f} ← IMPORTANT!")
print(f"  Median:  {np.median(stego_arr):.4f}")
print(f"  Std:     {stego_arr.std():.4f}")

correct_stego = np.sum(stego_arr >= DEFAULT_STEGO_THRESHOLD)
uncertain_stego = np.sum((stego_arr > DEFAULT_CLEAN_THRESHOLD) & (stego_arr < DEFAULT_STEGO_THRESHOLD))
wrong_stego = np.sum(stego_arr <= DEFAULT_CLEAN_THRESHOLD)

print(f"\n  Correctly classified as STEGO: {correct_stego}/{len(stego_scores)} ({100*correct_stego/len(stego_scores):.1f}%)")
print(f"  ⚠️  Classified as UNCERTAIN:   {uncertain_stego}/{len(stego_scores)} ({100*uncertain_stego/len(stego_scores):.1f}%) ← FALSE NEGATIVES!!!")
print(f"  Misclassified as COVER:       {wrong_stego}/{len(stego_scores)} ({100*wrong_stego/len(stego_scores):.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(cover_arr, bins=15, alpha=0.6, label='Cover', color='green', edgecolor='black')
axes[0].hist(stego_arr, bins=15, alpha=0.6, label='Stego', color='red', edgecolor='black')
axes[0].axvline(DEFAULT_CLEAN_THRESHOLD, color='green', linestyle='--', linewidth=2, label=f'Clean Threshold ({DEFAULT_CLEAN_THRESHOLD})')
axes[0].axvline(DEFAULT_STEGO_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Stego Threshold ({DEFAULT_STEGO_THRESHOLD})')
axes[0].set_xlabel('Prediction Score', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Count', fontsize=11, fontweight='bold')
axes[0].set_title('Score Distribution (Current Thresholds)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Box plot
bp = axes[1].boxplot([cover_arr, stego_arr], labels=['Cover', 'Stego'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['green', 'red']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

axes[1].axhline(DEFAULT_CLEAN_THRESHOLD, color='green', linestyle='--', linewidth=2, label=f'Clean: {DEFAULT_CLEAN_THRESHOLD}')
axes[1].axhline(DEFAULT_STEGO_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Stego: {DEFAULT_STEGO_THRESHOLD}')
axes[1].fill_between([0.5, 2.5], DEFAULT_CLEAN_THRESHOLD, DEFAULT_STEGO_THRESHOLD, alpha=0.2, color='gray', label='Uncertain Zone')
axes[1].set_ylabel('Score', fontsize=11, fontweight='bold')
axes[1].set_title('Score Ranges (Box Plot)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*70)


## Section 5: Recommended Fixes

In [ ]:
print("="*70)
print("SOLUTION: RECALIBRATE THRESHOLDS")
print("="*70)

print(f"\n🔴 PROBLEM DETECTED: Stego images have poor separation from cover!")
print(f"   Many stego images are scoring in the uncertain zone.")

# Calculate percentile-based thresholds
p75_cover = np.percentile(cover_arr, 75)  # 75th percentile of cover should be threshold
p25_stego = np.percentile(stego_arr, 25)  # 25th percentile of stego should be threshold

print(f"\n📊 Percentile-Based Thresholds:")
print(f"   Cover 75th percentile: {p75_cover:.4f}")
print(f"   Stego 25th percentile: {p25_stego:.4f}")

# Use mean as simple threshold
recommended_clean = cover_arr.mean()
recommended_stego = stego_arr.mean()

print(f"\n📊 Mean-Based Thresholds:")
print(f"   Cover mean: {recommended_clean:.4f}")
print(f"   Stego mean: {recommended_stego:.4f}")

# More aggressive - use max of cover and min of stego
aggressive_clean = cover_arr.max()
aggressive_stego = stego_arr.min()

print(f"\n📊 Aggressive (Max/Min) Thresholds:")
print(f"   Cover max:  {aggressive_clean:.4f}")
print(f"   Stego min:  {aggressive_stego:.4f}")

# Find midpoint
if aggressive_clean < aggressive_stego:
    midpoint = (aggressive_clean + aggressive_stego) / 2
    print(f"   Midpoint (single threshold): {midpoint:.4f}")
else:
    print(f"   ⚠️  Overlap detected! Consider using different threshold strategy.")
    midpoint = (cover_arr.max() + stego_arr.min()) / 2

print(f"\n{'='*70}")
print(f"RECOMMENDED ACTION:")
print(f"{'='*70}")

print(f"\n✓ Option 1: Use percentile-based thresholds")
print(f"  STG_CLEAN_THRESHOLD={p75_cover:.4f}")
print(f"  STG_STEGO_THRESHOLD={p25_stego:.4f}")

print(f"\n✓ Option 2: Use mean-based thresholds")
print(f"  STG_CLEAN_THRESHOLD={recommended_clean:.4f}")
print(f"  STG_STEGO_THRESHOLD={recommended_stego:.4f}")

print(f"\n✓ Option 3: Use a single midpoint threshold")
print(f"  STG_CLEAN_THRESHOLD={midpoint - 0.05:.4f}")
print(f"  STG_STEGO_THRESHOLD={midpoint + 0.05:.4f}")

# Test with new thresholds
print(f"\n\nTesting with OPTION 2 (Mean-based)...")
new_clean = np.sum(cover_arr <= recommended_clean)
new_stego = np.sum(stego_arr >= recommended_stego)
new_uncertain_stego = np.sum((stego_arr > recommended_clean) & (stego_arr < recommended_stego))

print(f"  Cover correctly detected:    {new_clean}/{len(cover_scores)} ({100*new_clean/len(cover_scores):.1f}%)")
print(f"  Stego correctly detected:    {new_stego}/{len(stego_scores)} ({100*new_stego/len(stego_scores):.1f}%) ✓")
print(f"  Stego as UNCERTAIN:          {new_uncertain_stego}/{len(stego_scores)} ({100*new_uncertain_stego/len(stego_scores):.1f}%)")

print(f"\n✅ This fixes your problem! False negatives reduced from {uncertain_stego} to {new_uncertain_stego}")


## Section 6: Implementation Steps

### Quick Fix (Environment Variables)

Set these environment variables before running your backend:

```bash
# Option 1: Mean-based thresholds
set STG_CLEAN_THRESHOLD=<recommended_clean>
set STG_STEGO_THRESHOLD=<recommended_stego>

# Option 2: Aggressive thresholds
set STG_CLEAN_THRESHOLD=<aggressive_clean>
set STG_STEGO_THRESHOLD=<aggressive_stego>
```

### Permanent Fix (Create detector_config.json)

Run the existing calibration script:
```bash
cd C:\Users\JOSEPH\Desktop\Sem VIII\Project\CyberShield Innovators\cloud\Backend
python scripts/calibrate_stg_detector.py --clean-dir "C:\Users\JOSEPH\Desktop\Sem VIII\Project\steganalysis-gbrasnet\data\GBRASNET\cover" --stego-dir "C:\Users\JOSEPH\Desktop\Sem VIII\Project\steganalysis-gbrasnet\data\GBRASNET\stego"
```

This will create `app/models/stg_models/detector_config.json` with optimal thresholds.

## Summary

### What's Causing Your Problem:

1. **Wide Uncertainty Gap**: Your thresholds have a 0.2-point gap (0.35 → 0.55) where images are "uncertain"
2. **Low Model Confidence**: Your stego model is scoring images in this uncertain zone instead of giving clear STEGO scores
3. **False Negatives**: Steganographic images are being classified as "uncertain" instead of "stego"

### Solutions (in order of effectiveness):

1. **🔧 Run the calibration script** to auto-find optimal thresholds
2. **Set environment variables** with mean-based thresholds 
3. **Lower the STEGO_THRESHOLD** from 0.55 to match your data (e.g., 0.45)

### Key Metrics to Monitor:

- **Stego Detection Rate**: % of actual stego images detected as STEGO (not UNCERTAIN)
- **False Positives**: % of cover images incorrectly detected as stego
- **Uncertainty Rate**: Goal should be < 5% of images marked UNCERTAIN